# Triage smoke test: the real chain on 14 invented reviews

Runs in about 10 minutes on a free Colab **GPU** runtime. A CPU runtime also works, but the suggestion model is
slow there. It checks that DistilBERT, Jev and Qwen run together, and shows what each stage says about a few
reviews. It is **not an evaluation**: there are no labels, no metrics and no thresholds
([README](https://github.com/krimits/hotel-review-nlp/blob/7da4eb83fff576c33a7a1ba492f29b0c0531fc07/docs/experiments/triage_smoke/README.md)).

- **Data.** The reviews are invented (`docs/experiments/triage_smoke/reviews.json`), so no guest's text is sent
  anywhere.
- **Jev, optional.** If the notebook finds a Colab secret named `OPENROUTER_API_KEY`, it also runs the arm with
  Jev, and the 14 invented reviews go to OpenRouter and to the provider behind it. Add the secret under the key
  icon in the left bar and switch on *Notebook access*. **Never paste the key into a cell, into this notebook or
  into a chat.** Without the secret, only the arm without Jev runs, and nothing leaves Colab.
- **Downloads.** The published DistilBERT and `Qwen/Qwen2.5-0.5B-Instruct`, about 1.3 GB in all.

**At the end** it shows the table with the problems and notes, and downloads `triage_smoke.zip`
(`summary.md`, `results.jsonl`, `run.json`). Send those, or the end of the run.

In [ ]:
# The commit that holds the smoke test. The run uses exactly this code.
REVISION = "7da4eb83fff576c33a7a1ba492f29b0c0531fc07"

!git clone --quiet https://github.com/krimits/hotel-review-nlp.git
%cd hotel-review-nlp
!git checkout --quiet {REVISION}
!git log --oneline -1
!pip install --quiet --no-deps -e .
!pip install --quiet fastapi "transformers==4.56.2" "accelerate==1.10.1"

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none (a CPU runtime works, but the suggestion model will be slow)")

## The key, if there is one

The key is read from Colab's Secrets and goes to the process as an environment variable. It is never printed.

In [ ]:
import os

USE_JEV = False
try:
    from google.colab import userdata

    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    USE_JEV = bool(os.environ["OPENROUTER_API_KEY"].strip())
except Exception as error:  # no such secret, or no notebook access to it
    print("No OPENROUTER_API_KEY secret:", type(error).__name__)
print("Jev arm:", "ON: the 14 invented reviews will be sent to OpenRouter" if USE_JEV
      else "off: nothing leaves Colab")

## Run it

The first review that reaches Qwen includes loading its weights, so it takes longer than the others.

In [ ]:
import subprocess

command = "python scripts/triage_smoke.py --output runs/triage_smoke"
if USE_JEV:
    command += " --allow-external-api --route openrouter"
print("$", command, flush=True)
exit_code = subprocess.run(command, shell=True).returncode
print("\nexit code", exit_code, "(0: the chain works. 1: read the problems above)")

## The summary, and the files to send back

In [ ]:
import zipfile

from google.colab import files

print(open("runs/triage_smoke/summary.md", encoding="utf-8").read())
with zipfile.ZipFile("triage_smoke.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ("summary.md", "results.jsonl", "run.json"):
        archive.write(f"runs/triage_smoke/{name}", arcname=name)
files.download("triage_smoke.zip")